In [1]:
#@title student data
from IPython.display import Javascript
colab_base = "https://colab.research.google.com/drive/1BVBGdNkEuqIgFp6XxmaV9NJy5c2sX3ln?usp=sharing"
Grupa = "niedotyczy" # @param ["niedotyczy", "13_45","15_30","17_15"]
Student_ID = "473616" # @param {"type":"string"}
Link_to_this_colab = "https://github.com/pawelFelcyn/nlp" # @param {"type":"string"}
Mail = "" # @param {"type":"string","placeholder":"Optional"}

if Link_to_this_colab == colab_base:
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))

#Task B01

## 🧩 Średni embedding zdania

W tym zadaniu zbudujesz **reprezentację zdania** jako średnią embeddingów, bazując na słowniku i macierzy embeddingów.

---

## 🎯 Cel

- przećwiczyć pracę z embeddingami,
- zobaczyć, jak z embeddingów słów zrobić embedding zdania.

---

## 📘 Kontekst

Embedding słowa to wektor liczb opisujący jego znaczenie.
Najprostsza reprezentacja zdania to **średnia embeddingów jego słów**.

---

## ✅ Twoje zadanie

1. Masz słownik `vocab` i tablicę `embeddings`, gdzie `embeddings[i]` to embedding słowa `vocab[i]`.
2. Zaimplementuj funkcję `sentence_to_embedding(sentence_words, vocab, embeddings)`, która:
   - przyjmuje listę słów (bez znaków interpunkcyjnych),
   - dla każdego słowa znajduje jego embedding,
   - zwraca **średni embedding** (średnia po wszystkich wektorach, po wymiarach).
3. Na końcu policz embedding zdania:

   `["kot", "je", "mleko"]`

   i zapisz go w zmiennej `final_answer` jako **jeden string**:
   liczby oddzielone przecinkami, bez spacji, zaokrąglone do 3 miejsc, np.:

   `0.125,-0.333,1.000`

Serwer porówna ten string ze swoim kluczem — musi być identyczny.



In [2]:
#@title dane

import numpy as np

vocab = ["kot", "je", "mleko", "pies", "wodę", "mięso"]
# embeddings[i] odpowiada vocab[i]
embeddings = np.array([
    [ 0.2,  0.1, -0.3,  0.4],   # kot
    [ 0.0,  0.3,  0.1,  0.0],   # je
    [-0.1,  0.2,  0.0,  0.3],   # mleko
    [ 0.3,  0.0, -0.2,  0.1],   # pies
    [-0.2,  0.1,  0.2,  0.0],   # wodę
    [ 0.1, -0.1,  0.3, -0.2],   # mięso
])

test_sentence = ["kot", "je", "mleko"]



In [4]:
#@title code

import numpy as np

def sentence_to_embedding(sentence_words, vocab, embeddings):
    """Zwraca średni embedding zdania:
    - sentence_words: lista słów, np. ["kot", "je", "mleko"]
    - vocab: lista słów
    - embeddings: tablica numpy, embeddings[i] odpowiada vocab[i]
    """
    embeddings_list = []
    for word in sentence_words:
        idx = vocab.index(word)
        word_embedding = embeddings[idx]
        embeddings_list.append(word_embedding)
    sentence_embedding = np.mean(embeddings_list, axis=0)
    return sentence_embedding



In [6]:
# @title Answer

emb = sentence_to_embedding(test_sentence, vocab, embeddings)

rounded = [f"{x:.3f}" for x in emb]
final_answer = ",".join(rounded)
print(final_answer)



0.033,0.200,-0.067,0.233


In [7]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskB01"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.033,0.200,-0.067,0.233
24 znaków
Wysłano!
Punkty za zadanie: 1


#Task B02

## 🧩 Unigram i bigram na tokenach

W tym zadaniu zbudujesz prosty model językowy oparty o unigramy i bigramy.

---

## 🎯 Cel

- policzyć częstości unigramów i bigramów,
- na tej podstawie wyznaczyć najbardziej prawdopodobne słowa następujące po danym tokenie.

---

## 📘 Kontekst

To naturalne rozwinięcie zadań z tokenizacją: mamy już sekwencje tokenów, teraz nauczymy się z nich robić prosty model.

---

## ✅ Twoje zadanie

1. Masz listę sekwencji tokenów `token_sequences` (już pocięte na subwordy).
2. Zaimplementuj funkcje:
   - `build_unigram_counts(sequences)` → zwraca słownik `{token: count}`,
   - `build_bigram_counts(sequences)` → zwraca słownik `{(t1, t2): count}`.
3. Zaimplementuj funkcję:
   - `most_probable_next(token, bigram_counts)`  
     która zwraca ten `t2`, dla którego `(token, t2)` ma największy `count`.
4. Na końcu:
   - policz najbardziej prawdopodobne słowo po `"kot"` i po `"pies"`,
   - zapisz wynik jako string w `final_answer`:

   `kot->TOKEN1;pies->TOKEN2`  (bez spacji).



In [8]:
#@title dane

token_sequences = [
    ["kot", "je", "mleko"],
    ["kot", "je", "rybę"],
    ["pies", "je", "mięso"],
    ["pies", "pije", "wodę"],
    ["kot", "pije", "mleko"]
]



In [9]:
#@title code

from collections import Counter

def build_unigram_counts(sequences):
    counts = Counter()
    for seq in sequences:
        counts.update(seq)
    return counts

def build_bigram_counts(sequences):
    counts = Counter()
    for seq in sequences:
        for i in range(len(seq) - 1):
            pair = (seq[i], seq[i+1])
            counts[pair] += 1
    return counts

def most_probable_next(token, bigram_counts):
    best_t2 = None
    best_cnt = -1
    for (t1, t2), cnt in bigram_counts.items():
        if t1 == token and cnt > best_cnt:
            best_cnt = cnt
            best_t2 = t2
    return best_t2



In [10]:
# @title Answer

bigram_counts = build_bigram_counts(token_sequences)

next_after_kot = most_probable_next("kot", bigram_counts)
next_after_pies = most_probable_next("pies", bigram_counts)

final_answer = f"kot->{next_after_kot};pies->{next_after_pies}"



In [11]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskB02"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


kot->je;pies->je
16 znaków
Wysłano!
Punkty za zadanie: 1


#Task B03

## 🧩 Generowanie sekwencji z modelu bigramowego

Na koniec wykorzystamy bigramy do wygenerowania krótkiej sekwencji tokenów.

---

## 🎯 Cel

- użyć policzonych bigramów do generowania tekstu,
- zrozumieć, jak działa prosta generacja języka.

---

## ✅ Twoje zadanie

1. Użyj tych samych `token_sequences` co w B02.
2. Zaimplementuj funkcję:
   `generate_sequence(start_token, bigram_counts, max_len=5)`:
   - zaczyna od `start_token`,
   - w każdej iteracji wybiera `most_probable_next` i dopisuje go do listy,
   - zatrzymuje się po osiągnięciu `max_len` lub gdy `most_probable_next` zwróci `None`.
3. W `final_answer` zapisz dwie sekwencje:
   - sekwencja od `"kot"`,
   - sekwencja od `"pies"`,

w formacie:

`kot:tok1 tok2 tok3;pies:tok1 tok2 tok3`



In [16]:
#@title dane

token_sequences = [
    ["mały", "kot", "spokojnie", "spał", "na", "słonecznym", "parapecie", "przez", "całe", "popołudnie"],
    ["wysoka", "trawa", "kołysała", "się", "delikatnie", "pod", "wieczornym", "chłodnym", "wiatrem"],
    ["stary", "pies", "wiernie", "czekał", "przed", "domem", "na", "swojego", "opiekuna", "codziennie"],
    ["ciemne", "chmury", "zbierały", "się", "nad", "miastem", "zapowiadając", "gwałtowną", "nocną", "burzę"],
    ["dzieci", "bawiły", "się", "radośnie", "na", "placu", "zabaw", "aż", "do", "zmroku"],
    ["długa", "droga", "prowadziła", "przez", "gęsty", "las", "pełen", "ptasich", "śpiewów", "rankiem"],
    ["nowy", "sąsiad", "powoli", "wprowadzał", "meble", "do", "mieszkania", "na", "trzecim", "piętrze"],
    ["pociąg", "przyjechał", "punktualnie", "pomimo", "złej", "pogody", "i", "silnego", "bocznego", "wiatru"],
    ["czerwone", "liście", "opadały", "lekko", "z", "drzew", "zwiastując", "początek", "późnej", "jesieni"],
    ["młody", "naukowiec", "pracował", "nad", "projektem", "w", "laboratorium", "do", "późnej", "nocy"],
    ["ptaki", "głośno", "śpiewały", "o", "świcie", "witając", "kolejny", "piękny", "letni", "poranek"],
    ["duży", "statek", "wpłynął", "powoli", "do", "portu", "po", "długiej", "morskiej", "podróży"],
    ["miły", "kelner", "podał", "nam", "ciepły", "obiad", "z", "domowymi", "dodatkami", "dzisiaj"],
    ["stare", "radio", "grało", "cicho", "w", "tle", "tworząc", "miłą", "rodzinną", "atmosferę"],
    ["zwinna", "wiewiórka", "skakała", "szybko", "po", "gałęziach", "szukając", "ukrytych", "zapasów", "orzechów"],
    ["młoda", "kobieta", "kupowała", "warzywa", "na", "targu", "przed", "niedzielnym", "rodzinnym", "obiadem"],
    ["ciemny", "las", "ożywał", "nocą", "pełen", "tajemniczych", "szmerów", "i", "echo", "kroków"],
    ["mały", "samolot", "leciał", "spokojnie", "nad", "górami", "w", "kierunku", "odległego", "miasta"],
    ["chłodny", "wiatr", "owiewał", "twórców", "stojących", "na", "plaży", "przy", "zachodzącym", "słońcu"],
    ["kierowca", "cierpliwie", "czekał", "w", "długim", "korku", "aż", "światła", "zmienią", "kolor"],
    ["stara", "biblioteka", "kryła", "książki", "z", "różnych", "epok", "pełne", "zapomnianych", "historii"],
    ["ogromny", "dąb", "stał", "dumnie", "na", "polanie", "od", "wielu", "setek", "lat"],
    ["mężczyzna", "naprawiał", "rower", "pod", "drzewem", "korzystając", "z", "dobrego", "porannego", "światła"],
    ["wodospad", "szumiał", "głośno", "spadając", "z", "wysokiej", "skalnej", "półki", "w", "dolinę"],
    ["uczniowie", "szli", "powoli", "do", "szkoły", "rozmawiając", "o", "nadchodzących", "egzaminach", "wiosennych"],
    ["mała", "łódka", "dryfowała", "spokojnie", "po", "jeziorze", "pod", "czystym", "błękitnym", "niebem"],
    ["autor", "pisał", "kolejny", "rozdział", "powieści", "siedząc", "samotnie", "przy", "dużym", "biurku"],
    ["kwiaty", "kwitły", "obficie", "na", "rabatach", "tworząc", "malownicze", "kolorowe", "dywany"],
    ["świeże", "pieczywo", "pachniało", "przyjemnie", "w", "niewielkiej", "lokalnej", "piekarni", "od", "rana"],
    ["rynek", "tętnił", "życiem", "pełen", "kupców", "turystów", "i", "głośnych", "rozmów"],
    ["słońce", "powoli", "znikało", "za", "horyzontem", "malując", "niebo", "złotymi", "barwami"],
    ["rowerzysta", "przejechał", "szybko", "przez", "most", "unikając", "nadjeżdżających", "samochodów", "z", "prawej"],
    ["aktor", "przygotowywał", "się", "do", "roli", "czytając", "scenariusz", "kilka", "godzin", "codziennie"],
    ["morze", "uderzało", "falami", "o", "skały", "tworząc", "biały", "pieniący", "się", "biały"],
    ["młode", "drzewka", "rosły", "szybko", "na", "żyznej", "glebie", "w", "wilgotnym", "klimacie"],
    ["sportowcy", "trenowali", "wytrwale", "na", "stadionie", "przygotowując", "się", "do", "ważnych", "zawodów"],
    ["kot", "wygrzewał", "się", "leniwie", "na", "dywanie", "po", "długiej", "nocnej", "wędrówce"],
    ["dziecko", "rysowało", "kolorowy", "obrazek", "siedząc", "przy", "małym", "drewnianym", "stoliku"],
    ["drobny", "deszcz", "spadał", "cicho", "na", "pustą", "ulicę", "o", "świcie"],
    ["jaskółki", "krążyły", "szybko", "nad", "łąką", "polując", "na", "drobne", "owady"],
    ["malarz", "pracował", "nad", "płótnem", "mieszając", "różne", "kolory", "na", "palecie"],
    ["robotnicy", "stawiali", "nowy", "most", "nad", "rzeką", "korzystając", "z", "ciężkiego", "sprzętu"],
    ["kwiaty", "pachniały", "słodko", "rozsiewając", "aromat", "po", "całym", "ogrodzie", "wiosennym"],
    ["listonosz", "roznosił", "paczki", "po", "dzielnicy", "pracując", "od", "rana", "do", "popołudnia"],
    ["tancerka", "ćwiczyła", "układ", "w", "studio", "przed", "ważnym", "pokazem", "tanecznym"],
    ["nad", "rzeką", "unosiła", "się", "mgła", "tworząc", "tajemniczy", "poranny", "klimat"],
    ["muzyk", "stroił", "instrument", "przed", "koncertem", "żeby", "uzyskać", "idealne", "brzmienie"],
    ["kobieta", "szukała", "prezentu", "dla", "przyjaciółki", "przechadzając", "się", "po", "sklepie"],
    ["kamienie", "toczyły", "się", "powoli", "ze", "skarpy", "po", "intensywnym", "deszczu"],
    ["mistrz", "kucharz", "przygotowywał", "kolację", "z", "wyjątkowych", "świeżych", "składników"],
    ["mężczyzna", "czytał", "gazetę", "siedząc", "w", "parku", "na", "drewnianej", "ławce"],
    ["sarenka", "przebiegła", "szybko", "przez", "ścieżkę", "zaskakując", "spacerujących", "ludzi"],
    ["samochody", "stały", "w", "korku", "czekając", "na", "zmianę", "świateł", "w", "centrum"],
    ["pies", "szukał", "patyka", "biegnąc", "radosnym", "krokiem", "po", "zielonej", "łące"],
    ["anioł", "na", "obrazie", "miał", "rozłożone", "skrzydła", "i", "spokojny", "wyraz"],
    ["samolot", "podchodził", "do", "lądownia", "z", "dużą", "ostrożnością", "pilota", "na", "pokładzie"],
    ["mężczyzna", "układał", "drewno", "przed", "zimą", "aby", "zapewnić", "ciepło", "całej", "rodzinie"],
    ["wiatrak", "obracał", "się", "wolno", "na", "szczycie", "wzgórza", "pod", "zachodzącym", "słońcem"],
    ["kocięta", "bawiły", "się", "sznurkiem", "przez", "kilkanaście", "minut", "bez", "przerwy"],
    ["student", "uczył", "się", "pilnie", "do", "egzaminu", "przeglądając", "notatki", "z", "wykładów"],
    ["góry", "wyłaniały", "się", "z", "mgły", "tworząc", "niezwykle", "malowniczy", "krajobraz"],
    ["płatki", "śniegu", "spadały", "powoli", "na", "ulicę", "pokrywając", "chodniki", "białym", "puchem"],
    ["trawa", "rosła", "gęsto", "na", "łące", "po", "ostatnich", "intensywnych", "deszczach"],
    ["aktorzy", "ćwiczyli", "scenę", "w", "teatrze", "przygotowując", "premierę", "na", "kolejny", "tydzień"],
    ["kot", "obserwował", "ptaki", "siedząc", "cierpliwie", "pod", "oknem", "w", "salonie"],
    ["zegar", "wybijał", "północ", "echo", "rozchodziło", "się", "po", "starych", "korytarzach"],
    ["turyści", "robili", "zdjęcia", "zachwycając", "się", "panoramą", "z", "górskiego", "szczytu"],
    ["dziecko", "skakało", "po", "kałużach", "ciesząc", "się", "z", "letniego", "deszczu"],
    ["przewodnik", "opowiadał", "historię", "miasta", "pokazując", "najstarsze", "zabytki", "grupie", "turystów"],
    ["łódź", "odpływała", "powoli", "od", "brzegu", "unosząc", "się", "na", "spokojnych", "falach"],
    ["drzewo", "szumiało", "cicho", "pod", "naporem", "delikatnego", "porannego", "wiatru"],
    ["sportowiec", "biegał", "codziennie", "rano", "żeby", "utrzymać", "wysoką", "formę", "treningową"],
    ["rolnik", "zbierał", "plony", "pracując", "od", "świtu", "aż", "do", "zmierzchu"],
    ["mama", "przygotowała", "pyszny", "obiad", "dla", "całej", "swojej", "rodziny", "dziś"],
    ["wędkarz", "złowił", "dużą", "rybę", "stojąc", "cierpliwie", "przy", "brzegu", "jeziora"],
    ["słońce", "oświetlało", "zielone", "pola", "rozciągające", "się", "aż", "po", "horyzont"],
    ["pan", "zagrał", "piękną", "melodię", "na", "skrzypcach", "podczas", "kameralnego", "koncertu"],
    ["kobieta", "podlewała", "kwiaty", "w", "ogrodzie", "dbając", "o", "różnorodne", "rośliny"],
    ["dzieci", "czytały", "książki", "siedząc", "w", "bibliotece", "pod", "opieką", "nauczyciela"],
    ["kot", "przechadzał", "się", "powoli", "po", "pokoju", "szukając", "miękkiego", "miejsca"],
    ["mężczyzna", "grał", "w", "szachy", "myśląc", "nad", "kolejnym", "trudnym", "ruchem"],
    ["pogoda", "zmieniała", "się", "gwałtownie", "zapowiadając", "zbliżającą", "się", "letnią", "burzę"],
    ["samica", "lwa", "polowała", "na", "sawannie", "żeby", "nakarmić", "swoje", "młode"],
    ["pilot", "sprawdził", "systemy", "przed", "startem", "dbając", "o", "bezpieczeństwo", "pasażerów"],
    ["muzeum", "oferowało", "wiele", "ekspozycji", "poświęconych", "sztuce", "i", "dawnej", "kulturze"],
    ["komputer", "działał", "sprawnie", "po", "aktualizacji", "dostarczając", "użytkownikom", "lepszą", "wydajność"],
    ["dźwięk", "fal", "uspokajał", "ludzi", "odpoczywających", "na", "szerokiej", "piaszczystej", "plaży"],
    ["pies", "merdał", "ogonem", "z", "radości", "widząc", "swojego", "właściciela", "wracającego"],
    ["kobieta", "szła", "spokojnie", "deptakiem", "słuchając", "ulubionej", "muzyki", "na", "słuchawkach"],
    ["kot", "zamykał", "oczy", "mrucząc", "z", "zadowolenia", "po", "smacznym", "posiłku"],
    ["artysta", "projektował", "nowe", "dzieło", "inspirowane", "naturą", "i", "wspomnieniami", "z", "podróży"],
    ["para", "tańczyła", "powoli", "na", "parkiecie", "w", "blasku", "kolorowych", "lamp"],
    ["rzeka", "płynęła", "szeroko", "pośród", "lasów", "tworząc", "piękne", "zakola", "na", "dolinie"]
]




In [17]:
#@title code

from collections import Counter

def build_bigram_counts(sequences):
    counts = Counter()
    for seq in sequences:
        for i in range(len(seq) - 1):
            pair = (seq[i], seq[i+1])
            counts[pair] += 1
    return counts

def most_probable_next(token, bigram_counts):
    best_t2 = None
    best_cnt = -1
    for (t1, t2), cnt in bigram_counts.items():
        if t1 == token and cnt > best_cnt:
            best_cnt = cnt
            best_t2 = t2
    return best_t2

def generate_sequence(start_token, bigram_counts, max_len=5):
    seq = [start_token]
    current = start_token
    for _ in range(max_len - 1):
        nxt = most_probable_next(current, bigram_counts)
        if nxt is None:
            break
        seq.append(nxt)
        current = nxt
    return seq



In [19]:
# @title Answer

bigram_counts = build_bigram_counts(token_sequences)

seq_kot = generate_sequence("mama", bigram_counts, max_len=10)
seq_pies = generate_sequence("rzeka", bigram_counts, max_len=10)

seq_kot_str = " ".join(seq_kot)
seq_pies_str = " ".join(seq_pies)

final_answer = f"mama:{seq_kot_str};rzeka:{seq_pies_str}"
print(final_answer)



mama:mama przygotowała pyszny obiad z drzew zwiastując początek późnej jesieni;rzeka:rzeka płynęła szeroko pośród lasów tworząc miłą rodzinną atmosferę


In [20]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskB04"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


mama:mama przygotowała pyszny obiad z drzew zwiastując początek późnej jesieni;rzeka:rzeka płynęła szeroko pośród lasów tworząc miłą rodzinną atmosferę
151 znaków
Wysłano!
Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny
